# AllTalk TTS Fine-tuning - Nicholas Ofczarek German Voice

This notebook fine-tunes the AllTalk TTS model on Nicholas Ofczarek's German voice using 169 training segments (~84 minutes).

## Step 1: Setup Environment

In [ ]:
# Install required packages with Python 3.11
!python3.11 -m pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!python3.11 -m pip install -q transformers TTS gradio faster-whisper pandas librosa pydub scipy
!python3.11 -m pip install -q trainer
print("✓ Dependencies installed")

In [ ]:
# Fix Python version (Colab uses 3.12, but TTS needs ≤3.11)
import sys
print(f"Current Python: {sys.version}")

# Downgrade to Python 3.11
!apt-get update -qq && apt-get install -qq -y python3.11 python3.11-dev python3.11-venv

# Set Python 3.11 as default
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1

# Verify
import subprocess
result = subprocess.run(['python3', '--version'], capture_output=True, text=True)
print(f"New Python: {result.stdout.strip()}")

## Step 2: Upload Training Data

Upload the `nicholas_ofczarek.zip` file containing your training data (169 audio segments + metadata)

In [ ]:
from google.colab import files
import zipfile
import os

# Upload the training data zip file
print("Please select nicholas_ofczarek.zip file...")
uploaded = files.upload()

# Extract the uploaded zip
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall()
        print(f"✓ Extracted {filename}")
        os.remove(filename)

# Verify the dataset
dataset_dir = '/content/nicholas_ofczarek'
if os.path.exists(dataset_dir):
    wav_files = [f for f in os.listdir(dataset_dir) if f.endswith('.wav')]
    print(f"✓ Found {len(wav_files)} audio files")
    if os.path.exists(os.path.join(dataset_dir, 'metadata.txt')):
        with open(os.path.join(dataset_dir, 'metadata.txt'), encoding='utf-8') as f:
            segments = f.readlines()
        print(f"✓ Metadata contains {len(segments)} segments")
else:
    print("✗ Dataset directory not found")

## Step 3: Clone AllTalk Repository

In [ ]:
import os
os.chdir('/content')

# Clone AllTalk
!git clone --depth 1 https://github.com/erew123/alltalk_tts.git alltalk
print("✓ AllTalk cloned")

## Step 4: Prepare Dataset for AllTalk Fine-tuning

In [ ]:
import shutil
from pathlib import Path

# Create AllTalk fine-tune input directory
alltalk_dataset_dir = Path('/content/alltalk/finetune/input/nicholas_ofczarek')
alltalk_dataset_dir.mkdir(parents=True, exist_ok=True)

# Copy our dataset
source_dir = Path('/content/nicholas_ofczarek')

# Copy audio files
for wav_file in source_dir.glob('*.wav'):
    shutil.copy2(wav_file, alltalk_dataset_dir / wav_file.name)

# Copy metadata
if (source_dir / 'metadata.txt').exists():
    shutil.copy2(source_dir / 'metadata.txt', alltalk_dataset_dir / 'metadata.txt')

# Verify
wav_count = len(list(alltalk_dataset_dir.glob('*.wav')))
print(f"✓ Prepared {wav_count} files in AllTalk format")
print(f"✓ Dataset ready at: {alltalk_dataset_dir}")

## Step 5: Run Fine-tuning (20 epochs)

This will take approximately 2-4 hours on Colab's GPU

In [ ]:
# DETAILED DEBUG - Test each component
import sys
import os
from pathlib import Path

os.chdir('/content/alltalk')
print("DETAILED DEBUG CHECK")
print("="*70)

# Test 1: Import TTS
print("\n[1] Testing TTS imports...")
try:
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts
    print("    ✓ TTS imports work")
except Exception as e:
    print(f"    ✗ TTS import failed: {e}")
    import traceback
    traceback.print_exc()

# Test 2: Check dataset
print("\n[2] Checking dataset...")
dataset_dir = Path('/content/alltalk/finetune/input/nicholas_ofczarek')
try:
    wav_files = list(dataset_dir.glob('*.wav'))
    metadata_file = dataset_dir / 'metadata.txt'
    print(f"    ✓ Found {len(wav_files)} WAV files")
    
    if metadata_file.exists():
        with open(metadata_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        print(f"    ✓ Metadata has {len(lines)} lines")
        print(f"    Sample line: {lines[0][:100]}")
    else:
        print("    ✗ Metadata.txt missing!")
except Exception as e:
    print(f"    ✗ Dataset check failed: {e}")

# Test 3: Check GPU
print("\n[3] Checking GPU...")
try:
    import torch
    print(f"    GPU available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"    GPU name: {torch.cuda.get_device_name(0)}")
        print(f"    GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
except Exception as e:
    print(f"    ✗ GPU check failed: {e}")

# Test 4: Try running finetune with verbose output
print("\n[4] Running finetune.py with output capture...")
print("="*70)

import subprocess
result = subprocess.run(
    [
        sys.executable, 'finetune.py',
        '--dataset_path', str(dataset_dir),
        '--language', 'de',
        '--output_model_dir', '/content/finetuned_models',
        '--epochs', '20'
    ],
    capture_output=True,
    text=True,
    timeout=14400
)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)
print(f"\nReturn code: {result.returncode}")

In [ ]:
# Debug: Check setup before fine-tuning
import os
from pathlib import Path

print("DEBUG: Pre-fine-tuning checks")
print("="*60)

# Check metadata format
metadata_file = alltalk_dataset_dir / 'metadata.txt'
if metadata_file.exists():
    with open(metadata_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[:3]
    print(f"✓ Metadata file exists ({len(lines)} lines shown):")
    for i, line in enumerate(lines):
        print(f"  Line {i}: {line[:80]}...")
else:
    print("✗ Metadata file missing!")

# Check audio files
wav_files = list(alltalk_dataset_dir.glob('*.wav'))
print(f"\n✓ Found {len(wav_files)} audio files")
print(f"  Sample: {wav_files[0].name if wav_files else 'None'}")

# Check write permissions
try:
    test_file = Path('/content/finetuned_models/test.txt')
    test_file.parent.mkdir(parents=True, exist_ok=True)
    test_file.write_text('test')
    test_file.unlink()
    print(f"\n✓ Output directory writable")
except Exception as e:
    print(f"\n✗ Cannot write to output dir: {e}")

print("="*60)

In [ ]:
import subprocess
import sys
os.chdir('/content/alltalk')

# Run fine-tuning with error output
print("Starting fine-tuning...")
print(f"Dataset: {alltalk_dataset_dir}")
print(f"Epochs: 20")
print(f"Language: German (de)")
print("="*60 + "\n")

try:
    result = subprocess.run(
        [
            sys.executable, 'finetune.py',
            '--dataset_path', str(alltalk_dataset_dir),
            '--language', 'de',
            '--output_model_dir', '/content/finetuned_models',
            '--epochs', '20'
        ],
        capture_output=False,
        text=True,
        timeout=14400  # 4 hour timeout
    )

    if result.returncode == 0:
        print("\n" + "="*60)
        print("✓ FINE-TUNING COMPLETED SUCCESSFULLY!")
        print("="*60)
    else:
        print(f"\n✗ Fine-tuning exited with code {result.returncode}")
except subprocess.TimeoutExpired:
    print("✗ Fine-tuning timed out after 4 hours")
except Exception as e:
    print(f"✗ Error: {e}")

## Step 6: Download Fine-tuned Model

In [ ]:
# Check if model was created
model_dir = Path('/content/finetuned_models')
if model_dir.exists():
    print(f"Model directory contents:")
    for item in model_dir.rglob('*'):
        if item.is_file():
            size_mb = item.stat().st_size / (1024*1024)
            print(f"  {item.relative_to(model_dir)}: {size_mb:.1f}MB")
    
    # Download
    print("\nDownloading model...")
    !cd /content && zip -r nicholas_ofczarek_model.zip finetuned_models/ && ls -lh nicholas_ofczarek_model.zip
    files.download('/content/nicholas_ofczarek_model.zip')
    print("✓ Model downloaded")
else:
    print("Model directory not found")

## Done!

Your fine-tuned Nicholas Ofczarek German TTS model is ready. Download it and extract into your AllTalk `models/` directory.